# 8.2 Feature Engineering

**Goal:** evaluate statistical feature-selection criteria without leaking validation information.

**Prerequisites:** Modules 0–4 and earlier notebooks in this module; advanced notebooks also build on earlier modules 5–8. Read the section assumptions before using a method.

**Runtime:** Python 3.10+ with NumPy, SciPy, and Matplotlib. Install in your chosen environment with `python3 -m pip install numpy scipy matplotlib`, then open in JupyterLab or VS Code with a Python kernel. Use **Restart Kernel and Run All**. Each notebook is self-contained, includes saved outputs, uses synthetic data and a fixed local random seed, and needs no downloads. Numerical simulations illustrate results; they do not prove them or validate real-world assumptions.

## Module navigation

1. [Data Preprocessing](1_Data-Preprocessing.ipynb)
2. [Feature Engineering](2_Feature-Engineering.ipynb)
3. [Model Evaluation Statistics](3_Model-Evaluation-Statistics.ipynb)
4. [Statistical Learning Concepts](4_Statistical-Learning-Concepts.ipynb)
5. [Advanced ML Statistics](5_Advanced-ML-Statistics.ipynb)

## Learning route

Read each definition and assumption, work through its calculation, run the code, then attempt the practice before viewing the solutions. Distinguish descriptive association, causal inference, and predictive evaluation throughout.

In [1]:
import math
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats, optimize, special, linalg
rng = np.random.default_rng(2026)
plt.rcParams.update({"figure.figsize": (7, 4), "axes.titlesize": 12, "font.size": 10})

## 1. Statistical Feature Selection and Feature Distributions

Feature selection chooses a subset of recorded predictors; feature engineering can also create transforms or interactions. Filter methods rank variables separately, wrapper methods evaluate subsets with a model, and embedded methods select through fitting. Supervised selection uses labels and must occur within training folds. Even unsupervised selection can leak distribution information if fitted on the full dataset.

Inspect support, units, missingness, rare categories, subgroup coverage, and train/deployment distributions first. Stable marginal statistics do not guarantee stable joint relationships. A variable useful in-sample may be a post-outcome proxy or unavailable at inference.

## 2. Correlation-Based Selection and Variance Threshold

Pearson correlation detects linear association; Spearman detects monotone rank association. Low marginal correlation can hide interactions or nonlinear prediction. Highly correlated predictors can be redundant for a linear model, but dropping them can also remove useful distinct information. Arbitrary pairwise thresholds require validation.

Variance thresholds remove near-constant features. Units matter: a feature measured in millimeters has 10⁶ times the variance of the same feature in meters. A rare binary event has low variance p(1−p) yet can be decisive for fraud. This is a computational filter, not a universal relevance test.

In [2]:
n=200
x=rng.normal(size=n);noise=rng.normal(size=n);quadratic=x*x+rng.normal(0,.1,n)
X=np.column_stack([x,noise,np.ones(n)]);y=quadratic
print("Marginal correlations:",[np.corrcoef(X[:,j],y)[0,1] for j in [0,1]])
print("Feature variances:",X.var(0),"nonconstant mask:",X.var(0)>1e-12)

Marginal correlations: [np.float64(0.05660938860714565), np.float64(-0.01971018017660784)]
Feature variances: [1.19127807 0.97776271 0.        ] nonconstant mask: [ True  True False]


## 3. Chi-square Feature Selection and ANOVA F-test

A categorical chi-square independence test compares a feature/target contingency table against expected counts under independence. Sparse expected cells, dependent units, and many simultaneous screens need care. Some ML chi-square selectors use **nonnegative feature counts** rather than discretized contingency categories; negative standardized values are invalid for that count interpretation. Binning continuous predictors loses information and adds boundary choices.

An ANOVA F-test can screen a numerical feature across classes, testing equal class means under the classical model. It does not test every possible distribution difference, and equal means can coexist with distinct predictive shapes. A low p-value is not an effect size or a cross-validated usefulness guarantee.

In [3]:
labels=(x>0).astype(int)
table=np.array([[np.sum((x<0)&(labels==0)),np.sum((x<0)&(labels==1))],
                [np.sum((x>=0)&(labels==0)),np.sum((x>=0)&(labels==1))]])
print("Chi-square feature/label screen:",stats.chi2_contingency(table,correction=False).pvalue)
print("Class-mean ANOVA for x:",stats.f_oneway(x[labels==0],x[labels==1]))

Chi-square feature/label screen: 2.0884875837625688e-45
Class-mean ANOVA for x: F_onewayResult(statistic=np.float64(355.11185109467436), pvalue=np.float64(4.780253283745651e-46))


## 4. Mutual Information and Information Gain

For discrete X,Y, $I(X;Y)=\sum_{x,y}p(x,y)\log[p(x,y)/(p(x)p(y))]$. It is nonnegative in the population, zero iff independence, and can detect nonmonotone associations. Information gain in a decision-tree split is the reduction in target entropy; for a discrete split variable it equals mutual information.

The log base sets units: base 2 gives bits. Empirical plug-in MI is generally biased upward, especially with many categories relative to observations. Continuous-variable estimators need density, neighbor, or binning choices, and scores from different estimators may not be directly comparable. Maximizing a noisy MI estimate over many candidates also selects noise.

**Example:** bin a squared predictor relation and compute plug-in MI. The result depends on bins and is descriptive; it is not a bias-corrected estimate or a significance test.

In [4]:
def discrete_mi(a,b):
    _,ai=np.unique(a,return_inverse=True);_,bi=np.unique(b,return_inverse=True)
    counts=np.zeros((ai.max()+1,bi.max()+1));np.add.at(counts,(ai,bi),1)
    p=counts/counts.sum();expected=p.sum(1)[:,None]*p.sum(0)[None,:];mask=p>0
    return np.sum(p[mask]*np.log2(p[mask]/expected[mask]))
xb=np.digitize(x,np.quantile(x,[.2,.4,.6,.8]));yb=np.digitize(y,np.quantile(y,[.2,.4,.6,.8]))
print("Binned quadratic relationship MI (bits):",discrete_mi(xb,yb))
print("Binned independent-noise MI estimate:",discrete_mi(np.digitize(noise,[-1,0,1]),yb))
assert discrete_mi(xb,yb)>=-1e-12

Binned quadratic relationship MI (bits): 1.1737103370613682
Binned independent-noise MI estimate: 0.09663194542572293


## AI/ML application and interpretation

Use feature filters to reduce computation and generate hypotheses, then evaluate the selected pipeline independently. If reporting feature significance after selection, ordinary p-values that ignore the search are generally invalid. Preserve domain constraints, availability timing, and subgroup coverage alongside statistical scores.

## Practice

1. Why can a zero Pearson correlation hide a predictive feature?
2. Why does supervised feature selection belong inside cross-validation folds?
3. Is low feature variance proof of irrelevance?
4. Does larger empirical MI automatically mean a better deployment feature?

## Solutions

1. Nonlinear relations and interactions can have little or no linear marginal association.
2. Full-data selection exposes validation labels to the pipeline and biases evaluation.
3. No. Rare events can be strongly informative, and variance is unit-dependent.
4. No. Estimation bias, sample size, cardinality, leakage, and distribution shift all matter; validate the complete procedure.

**Readiness check:** identify the estimand, independent unit, assumptions, uncertainty source, and an important failure mode before using these methods.

**Next:** [Continue studying](3_Model-Evaluation-Statistics.ipynb).